# Tel Aviv Street Closures Ingestion - Bronze Layer

## Overview
This notebook implements the **Bronze (Raw) layer** ingestion for Tel Aviv street closure data from CSV files stored in Google Cloud Storage. This data pipeline processes street closure information for business compensation calculations.

## Purpose
* Ingest street closure data from CSV files in Databricks Volumes
* Store raw data in immutable bronze layer with full audit trail
* Generate deterministic event IDs for data lineage tracking
* Preserve original payload structure as JSON for schema-on-read processing

## Key Features
* **CSV-to-JSON Payload**: Each CSV row is preserved as a JSON string in the `payload` column
* **Event-Based Tracking**: SHA-256 hash of payload generates deterministic `event_id` for lineage
* **Audit Metadata**: Captures ingestion timestamp and source system identifier
* **Volume Integration**: Reads from Databricks Volumes (`/Volumes/workspace/default/external/`)
* **Data Validation**: Asserts CSV is not empty before processing
* **Error Handling**: Try-catch wrapper for graceful failure reporting
* **Idempotent Ingestion**: Merge-based approach prevents duplicate records on reruns

## Data Source
**Street Closures CSV**: `/Volumes/workspace/default/external/rechov_sagur.csv`

## Output Table
**Target**: `workspace.tel_aviv_municipality_raw.street_closures`

**Schema**:
| Column | Type | Description |
|--------|------|-------------|
| `uuid` | STRING | Unique record identifier (generated per row) |
| `payload` | STRING | Original CSV row as JSON string |
| `event_id` | STRING | SHA-256 hash of payload (deterministic deduplication key) |
| `ingestion_at` | TIMESTAMP | Time of ingestion |
| `source_system` | STRING | Fixed value: "CSV_Upload_GCS" |
| `file_name` | STRING | Source CSV filename for lineage tracking |

## Ingestion Strategy
**Mode**: Idempotent merge-based ingestion
* First run: Creates table with all CSV rows
* Subsequent runs: Merges based on `event_id` - only new records are inserted
* Prevents duplicate data on reruns
* Supports incremental processing patterns

## Design Principles
* **Immutable Raw Data**: Preserve original CSV structure in JSON format
* **Uniform Schema**: Matches API ingestion pattern (same metadata columns)
* **Schema-on-Read**: Minimal transformation - parsing happens in Silver layer
* **Lineage Tracking**: Deterministic event IDs enable end-to-end data tracking
* **Idempotency**: Safe to rerun without creating duplicates

In [0]:
import json
import uuid
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable

In [0]:
def ingest_closures_bronze_payload():
    """
    Ingests street closure data from a CSV file into the Bronze Delta table.
    
    Logic:
    1. Reads a CSV from Unity Catalog Volumes.
    2. Encapsulates each row into a JSON payload to match the bronze 'Schema-on-Read' pattern.
    3. Generates a deterministic 'event_id' using a SHA-256 hash of the payload.
    4. Performs an idempotent MERGE: Inserts records if the event_id is new, 
       otherwise ignores them to prevent duplicates.
    
    Metadata Added:
    - uuid: Unique identifier for the ingestion trace.
    - ingestion_at: Timestamp of processing.
    - source_system: Origin of the data (CSV).
    - file_name: Name of the source file for lineage tracking.
    - event_id: Hash for deduplication.
    
    Returns:
        str: A status message indicating success or the specific error encountered.
    """    
    # Configuration
    csv_path = "/Volumes/workspace/default/external/rechov_sagur.csv" 
    destination_table = "workspace.tel_aviv_municipality_raw.street_closures"
    
    # Extract just the file name from the path for cleaner metadata
    file_name = csv_path.split("/")[-1]

    try:
        # Read the CSV
        df_csv = spark.read.option("header", "true").csv(csv_path)
        rows_as_dicts = [row.asDict() for row in df_csv.collect()]
        
        assert len(rows_as_dicts) > 0, "CSV file is empty or could not be read."

        # Map to Row structure
        raw_data = [
            Row(
                uuid=str(uuid.uuid4()), 
                payload=json.dumps(r) 
            ) 
            for r in rows_as_dicts
        ]

        # Create DataFrame and Add Metadata
        df_new = spark.createDataFrame(raw_data) \
            .withColumn("ingestion_at", F.current_timestamp()) \
            .withColumn("source_system", F.lit("CSV_Upload_GCS")) \
            .withColumn("file_name", F.lit(file_name)) \
            .withColumn("event_id", F.sha2(F.col("payload"), 256))

        # Perform delta Merge 
        if spark.catalog.tableExists(destination_table):
            target_table = DeltaTable.forName(spark, destination_table)
            
            (target_table.alias("target")
             .merge(
                 df_new.alias("updates"), 
                 "target.event_id = updates.event_id"
             )
             .whenNotMatchedInsertAll()
             .execute())
            
            return f"Success: Merged data from {file_name} into {destination_table}."
        
        else:
            # For first time setup
            df_new.write.format("delta").mode("overwrite").saveAsTable(destination_table)
            return f"Success: Created {destination_table} with data from {file_name}."

    except Exception as e:
        return f"Error occurred during CSV ingestion: {e}"



In [0]:
# Execute
print(ingest_closures_bronze_payload())